# Qualidade de `tickets_suporte.csv`

Diagnóstico dos chamados ao suporte pelos mesmos cinco critérios usados nos outros arquivos, mais a checagem de privacidade do texto livre:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Respostas vazias ou sem conteúdo, e quem deveria estar no arquivo |
| **Validade** | Categorias, datas e números no domínio esperado? Texto livre que na prática é categoria converte? |
| **Consistência** | O texto concorda com as categorias e com os outros arquivos? |
| **Privacidade** | Há e-mail, telefone, CPF ou @ de terceiros no texto? |
| **Unicidade** | Cada chamado aparece uma vez só? |
| **Temporalidade** | As datas estão no período certo, antes do corte, no ritmo da base? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).
Transformar o texto em variáveis (extração com IA generativa, objetivo 5 do case) é outra etapa e não é feito aqui.

**Decisões já registradas que valem aqui** (`decisoes.md`): contas de teste e cópias de `usuarios.csv` removidas; datas
dos logs em horário de Brasília; nenhuma informação posterior ao corte. A falha de GPS (04/05 a 28/06/2026) foi registrada no diagnóstico de atividades (notebook 04).

**Privacidade:** texto livre aparece só como contagem e respostas mais frequentes, nunca ao lado do `user_id`.

In [1]:
ARQUIVO = "tickets_suporte.csv"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO, "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    # status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

# respostas sem conteúdo (vazio disfarçado), pelo checklist da skill
SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)   # só emoji ou pontuação
    return t.isin(SEM_CONTEUDO) | so_simbolo

PII = {
    "e-mail": r"[\w.+-]+@[\w-]+\.[a-z]{2,}",
    "telefone": r"\(?\b\d{2}\)?\s?9?\d{4}-?\d{4}\b",
    "CPF": r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
    "@ de perfil": r"(?<![\w.])@\w{3,}",
}

def contar_pii(s):
    return {nome: int(s.str.contains(rx, regex=True, case=False).sum()) for nome, rx in PII.items()}

brutos_usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False, usecols=["user_id", "email"])
email = brutos_usuarios.set_index("user_id")["email"].str.strip()
TESTE = set(email.index[email.str.lower().str.endswith("@vitaliza.com.br")])
COPIA = set(email.index[email != email.str.lower()])
usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

def contas_invalidas(df, criterio="Completude"):
    n_teste, n_copia = int(df["user_id"].isin(TESTE).sum()), int(df["user_id"].isin(COPIA).sum())
    sem_cad = int((~df["user_id"].isin(email.index)).sum())
    achado(criterio, "linhas de contas de teste, cópias ou sem cadastro",
           f"teste: {n_teste}; cópias: {n_copia}; sem cadastro: {sem_cad}", n_teste + n_copia + sem_cad, "Se houver: excluir")

## 0. Leitura

In [2]:
tk = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(tk) == 4_422  # cabeçalho da seção 13 do dicionário
N = len(tk)
print(f"chamados: {N:,} linhas, {tk['user_id'].nunique():,} contas -> {list(tk.columns)}")
tk["aberto"] = pd.to_datetime(tk["aberto_em"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
tk["resolvido"] = pd.to_datetime(tk["resolvido_em"].replace("", None), format="%Y-%m-%d", errors="coerce")
tk["mes"] = tk["aberto"].dt.to_period("M")
tk[["canal", "categoria_atendente", "status"]].value_counts().head(5)

chamados: 4,422 linhas, 3,989 contas -> ['ticket_id', 'user_id', 'aberto_em', 'canal', 'texto_cliente', 'categoria_atendente', 'status', 'resolvido_em']


canal  categoria_atendente  status   
chat   cancelamento         resolvido    391
       conta                resolvido    270
       tecnico              resolvido    206
loja   cancelamento         resolvido    199
email  cancelamento         resolvido    194
Name: count, dtype: int64

## 1. Completude

`resolvido_em` só existe nos chamados resolvidos (vazio estrutural nos abertos e sem resposta).

In [3]:
vazio = int((tk["texto_cliente"].str.strip() == "").sum())
lixo = int(sem_conteudo(tk["texto_cliente"]).sum())
achado("Completude", "texto do cliente vazio ou sem conteúdo", f"{vazio} e {lixo}", vazio + lixo, "Nenhum")
display(pd.crosstab(tk["status"], tk["resolvido_em"].eq("").map({True: "sem resolvido_em", False: "com resolvido_em"})))
estrutural = int((tk["status"] != "resolvido").sum())
achado("Completude", "resolvido_em vazio nos chamados não resolvidos", f"{estrutural:,} (aberto ou sem_resposta)", 0,
       "Nenhum: vazio estrutural", status="esperado")
contas_invalidas(tk)

resolvido_em,com resolvido_em,sem resolvido_em
status,,
aberto,0,981
resolvido,2917,0
sem_resposta,0,524


## 2. Validade

In [4]:
DOMINIO = {"canal": {"chat", "email", "loja"}, "status": {"resolvido", "aberto", "sem_resposta"},
           "categoria_atendente": {"tecnico", "cancelamento", "conta", "cobranca", "conteudo", "duvida_geral"}}
fora = {c: sorted(set(tk[c]) - v) for c, v in DOMINIO.items()}
print("fora do domínio:", fora, "| categorias observadas:", sorted(set(tk["categoria_atendente"])))
achado("Validade", "canal, status ou categoria fora do domínio", f"{fora}", sum(len(v) for v in fora.values()), "Nenhum")
falha = int(tk["aberto"].isna().sum() + (tk["resolvido"].isna() & (tk["resolvido_em"] != "")).sum())
achado("Validade", "aberto_em ou resolvido_em que não converte", f"{falha}", falha, "Nenhum")

fora do domínio: {'canal': [], 'status': [], 'categoria_atendente': []} | categorias observadas: ['cancelamento', 'cobranca', 'conta', 'conteudo', 'duvida_geral', 'tecnico']


## 3. Consistência

### Status, datas e cadastro

In [5]:
incoerente = int(((tk["status"] == "resolvido") != tk["resolvido"].notna()).sum())
achado("Consistência", "status resolvido sem data (ou data sem resolvido)", f"{incoerente}", incoerente, "Nenhum")
antes = int((tk["resolvido"] < tk["aberto"].dt.normalize()).sum())
achado("Consistência", "resolvido antes de aberto", f"{antes}", antes, "Nenhum")
print("dias até resolver:", (tk["resolvido"] - tk["aberto"].dt.normalize()).dt.days.describe().round(1).to_dict())
antes_cad = int((tk["aberto"].dt.normalize() < tk["user_id"].map(usuarios["data_cadastro"])).sum())
achado("Consistência", "chamado antes do dia do cadastro", f"{antes_cad}", antes_cad, "Nenhum")

dias até resolver: {'count': 2917.0, 'mean': 3.9, 'std': 3.8, 'min': 1.0, '25%': 1.0, '50%': 3.0, '75%': 5.0, 'max': 19.0}


### Categoria do atendente contra o texto do cliente

O mesmo texto se repete em muitos chamados (303 textos distintos). Se a categoria fosse confiável, cada texto teria quase
sempre a mesma categoria. Para cada texto, a categoria mais frequente é a "categoria do texto"; os demais chamados com
aquele texto divergem.

In [6]:
moda = tk.groupby("texto_cliente")["categoria_atendente"].agg(lambda s: s.value_counts().index[0])
participacao = tk.groupby("texto_cliente")["categoria_atendente"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
tk["categoria_do_texto"] = tk["texto_cliente"].map(moda)
diverge = tk["categoria_atendente"] != tk["categoria_do_texto"]
print(f"textos distintos: {tk['texto_cliente'].nunique()}; participação da categoria mais frequente por texto: "
      f"mediana {participacao.median():.0%}, mínimo {participacao.min():.0%}")
display(pd.crosstab(tk["categoria_do_texto"], tk["categoria_atendente"], margins=True))
print("divergência por mês:", diverge.groupby(tk["mes"]).mean().mul(100).round(0).to_dict())
print("divergência por canal:", diverge.groupby(tk["canal"]).mean().mul(100).round(0).to_dict())
achado("Consistência", "categoria do atendente diferente da categoria do texto",
       f"{diverge.sum():,} chamados ({100 * diverge.mean():.0f}%); a divergência é espalhada por todas as categorias, meses e canais "
       f"(a categoria mais frequente de cada texto aparece em {1 - diverge.mean():.0%} dos chamados); maior em maio e junho de 2026",
       int(diverge.sum()),
       "Não usar a categoria do atendente sozinha: classificar o texto do cliente (extração com IA generativa ou a categoria "
       "mais frequente do texto, como neste notebook)")

textos distintos: 303; participação da categoria mais frequente por texto: mediana 100%, mínimo 33%


categoria_atendente,cancelamento,cobranca,conta,conteudo,duvida_geral,tecnico,All
categoria_do_texto,,,,,,,
cancelamento,660,34,32,40,46,46,858
cobranca,78,450,57,64,62,67,778
conta,31,28,536,32,39,39,705
conteudo,16,21,11,314,26,13,401
duvida_geral,18,12,19,22,262,15,348
tecnico,110,106,102,116,117,781,1332
All,913,651,757,588,552,961,4422


divergência por mês: {Period('2025-07', 'M'): 30.0, Period('2025-08', 'M'): 23.0, Period('2025-09', 'M'): 31.0, Period('2025-10', 'M'): 32.0, Period('2025-11', 'M'): 34.0, Period('2025-12', 'M'): 23.0, Period('2026-01', 'M'): 28.0, Period('2026-02', 'M'): 27.0, Period('2026-03', 'M'): 31.0, Period('2026-04', 'M'): 26.0, Period('2026-05', 'M'): 41.0, Period('2026-06', 'M'): 36.0, Period('2026-07', 'M'): 30.0, Period('2026-08', 'M'): 29.0, Period('2026-09', 'M'): 25.0}
divergência por canal: {'chat': 31.0, 'email': 32.0, 'loja': 34.0}


## 4. Privacidade

In [7]:
pii = contar_pii(tk["texto_cliente"])
print(pii)
achado("Privacidade", "e-mail, telefone, CPF ou @ no texto", f"{pii}", sum(pii.values()), "Se houver: mascarar antes de qualquer envio a IA")

{'e-mail': 0, 'telefone': 0, 'CPF': 0, '@ de perfil': 0}


## 5. Unicidade

O mesmo cliente pode reabrir o mesmo problema. Duplicata seria o mesmo texto, da mesma conta, no mesmo minuto.

In [8]:
achado("Unicidade", "ticket_id repetido", f"{int(tk['ticket_id'].duplicated().sum())}", int(tk["ticket_id"].duplicated().sum()), "Nenhum")
mesmo_min = int(tk.duplicated(["user_id", "texto_cliente", "aberto"]).sum())
achado("Unicidade", "mesma conta, texto e minuto", f"{mesmo_min}", mesmo_min, "Se houver: manter um")
t = tk.sort_values(["user_id", "texto_cliente", "aberto"])
gap = t.groupby(["user_id", "texto_cliente"])["aberto"].diff().dt.days
reaberto = int((gap <= 7).sum())
print(f"mesmo texto da mesma conta em até 7 dias: {reaberto}; contas com mais de um chamado: {(tk['user_id'].value_counts() > 1).sum()}")
achado("Unicidade", "mesmo problema reaberto pela mesma conta em até 7 dias", f"{reaberto}", reaberto,
       "Nenhum: cada chamado é um contato; para contar problemas por conta, juntar os do mesmo texto em 7 dias",
       status="esperado" if reaberto else "ok")

mesmo texto da mesma conta em até 7 dias: 0; contas com mais de um chamado: 401


## 6. Temporalidade

Chamados de 01/07/2025 até o corte. O status e a data de resolução são os da exportação.

In [9]:
print(f"aberto_em de {tk['aberto'].min()} a {tk['aberto'].max()}")
fora_periodo = int(((tk["aberto"] < pd.Timestamp("2025-07-01")) | (tk["aberto"] >= FIM_CORTE)).sum())
achado("Temporalidade", "chamado aberto fora de 01/07/2025 a 30/09/2026", f"{fora_periodo}", fora_periodo, "Nenhum")
depois = tk["resolvido"] > CORTE
print("resolvidos depois do corte:", int(depois.sum()), "| até:", tk.loc[depois, "resolvido"].max())
achado("Temporalidade", "resolução depois do corte (status da exportação)",
       f"{int(depois.sum())} chamados resolvidos de 01/10 a {tk.loc[depois, 'resolvido'].max():%d/%m/%Y}: no corte estavam abertos",
       int(depois.sum()), "Recalcular o status no corte: resolvido só se resolvido_em <= 30/09/2026; senão, aberto (ou sem_resposta)")

aberto_em de 2025-07-01 08:15:00 a 2026-09-30 21:46:00
resolvidos depois do corte: 41 | até: 2026-10-07 00:00:00


### O salto de maio e junho de 2026

Hipótese: é a falha de GPS de 04/05 a 28/06/2026 (diagnóstico de atividades). Se for, o salto está nos chamados técnicos
sobre GPS, percurso e treinos perdidos ("perdi meus treinos da semana, o app travou no meio" só aparece nesses dois meses).

In [10]:
GPS = r"gps|percurso|rota|distância|distancia|travou|perdi meus treinos"
tk["sobre_gps"] = tk["texto_cliente"].str.lower().str.contains(GPS, regex=True)
mensal = pd.DataFrame({"chamados": tk.groupby("mes").size(), "sobre GPS, percurso ou treino perdido": tk.groupby("mes")["sobre_gps"].sum()})
mensal["outros"] = mensal["chamados"] - mensal["sobre GPS, percurso ou treino perdido"]
display(mensal)
PICO = [pd.Period("2026-05", "M"), pd.Period("2026-06", "M")]
pico = mensal.loc[PICO]
base = mensal.drop(index=PICO)
achado("Temporalidade", "salto de chamados em maio e junho de 2026",
       f"{int(pico['chamados'].sum()):,} chamados nos dois meses; os sobre GPS, percurso ou treino perdido passam de "
       f"{base['sobre GPS, percurso ou treino perdido'].mean():.0f} para {pico['sobre GPS, percurso ou treino perdido'].mean():.0f} "
       f"por mês e os demais ficam em {pico['outros'].mean():.0f} (base {base['outros'].mean():.0f}; cancelamento e troca de plano "
       "sobem um pouco): é a falha de GPS de 04/05 a 28/06",
       int(pico["chamados"].sum() - 2 * base["chamados"].mean()),
       "Nenhum: é um acontecimento real; nas análises de chamados, separar o período da falha de GPS", status="esperado")
horas = tk["aberto"].dt.hour
print("hora de abertura:", horas.value_counts().sort_index().to_dict())
achado("Temporalidade", "fuso de aberto_em (o dicionário não diz)",
       f"chamados abertos das {horas.min()}h às {horas.max()}h, horário comercial em Brasília (em UTC, seriam das {horas.min() - 3}h às {horas.max() - 3}h)",
       0, "Usar aberto_em como horário de Brasília")

,chamados,"sobre GPS, percurso ou treino perdido",outros
mes,,,
2025-07,240,0,240
2025-08,207,0,207
2025-09,173,0,173
2025-10,170,0,170
2025-11,199,0,199
2025-12,194,0,194
2026-01,178,0,178
2026-02,186,0,186
2026-03,248,0,248


hora de abertura: {8: 298, 9: 313, 10: 328, 11: 322, 12: 309, 13: 308, 14: 306, 15: 342, 16: 321, 17: 294, 18: 339, 19: 335, 20: 316, 21: 291}


## Resumo

In [11]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / f"qualidade_{Path(ARQUIVO).stem}.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 13, 'esperado': 2, 'atenção': 2}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,texto do cliente vazio ou sem conteúdo,ok,0 e 0,0,Nenhum,0.00
1,Completude,resolvido_em vazio nos chamados não resolvidos,esperado,"1,505 (aberto ou sem_resposta)",0,Nenhum: vazio estrutural,0.00
2,Completude,"linhas de contas de teste, cópias ou sem cadastro",ok,teste: 0; cópias: 0; sem cadastro: 0,0,Se houver: excluir,0.00
3,Validade,"canal, status ou categoria fora do domínio",ok,"{'canal': [], 'status': [], 'categoria_atendente': []}",0,Nenhum,0.00
4,Validade,aberto_em ou resolvido_em que não converte,ok,0,0,Nenhum,0.00
5,Consistência,status resolvido sem data (ou data sem resolvido),ok,0,0,Nenhum,0.00
6,Consistência,resolvido antes de aberto,ok,0,0,Nenhum,0.00
7,Consistência,chamado antes do dia do cadastro,ok,0,0,Nenhum,0.00
8,Consistência,categoria do atendente diferente da categoria do texto,atenção,"1,419 chamados (32%); a divergência é espalhada por todas as categ...",1419,Não usar a categoria do atendente sozinha: classificar o texto do ...,32.09
9,Privacidade,"e-mail, telefone, CPF ou @ no texto",ok,"{'e-mail': 0, 'telefone': 0, 'CPF': 0, '@ de perfil': 0}",0,Se houver: mascarar antes de qualquer envio a IA,0.00


In [12]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
